# Build an MCP Server for Your Notes -- tool logic only (Colab/Kaggle)

**Scope of this notebook: narrow, on purpose.** The [main lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/mcp-notes-server) explains why Colab and Kaggle are *not* a good fit for the actual MCP server project -- an MCP server is a standalone local process that a desktop AI client (Claude Desktop) launches and talks to directly over stdio, and neither Colab nor Kaggle can offer that. That's still true, and this notebook does not attempt to work around it.

What this notebook *does* let you do: experiment with the three **tool functions themselves** -- `search_notes`, `get_note_by_title`, and `list_recent_notes` -- as plain Python, with no `@mcp.tool()` decorator, no `FastMCP` server object, and no MCP protocol involved at all. This is useful for understanding and tweaking the *logic* the real server wraps, in a free hosted notebook, before (or instead of) setting anything up locally.

This is **not** a substitute for the real project. To get an actual MCP server running and connected to Claude Desktop, follow the [full lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/mcp-notes-server) locally with `uv`.

## Write a few sample notes to disk

The real project indexes [`examples/mcp-notes-server/notes/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/mcp-notes-server/notes) in the course repo -- 7 real notes covering recipes, book notes, and more. To keep this notebook self-contained (no repo clone needed), we write a small subset of those same notes to a temporary folder here instead.

In [ ]:
from pathlib import Path

NOTES_DIR = Path("/tmp/notes")
NOTES_DIR.mkdir(exist_ok=True)

sample_notes = {
    "weeknight-pasta.md": """# Weeknight Garlic Pasta

date: 2026-06-02
tags: recipe, quick, pasta

A 20-minute dinner for nights when nothing sounds good. Slice garlic thin,
cook it low and slow in olive oil until golden (not brown), add chili
flakes and a ladle of starchy pasta water, then toss in the drained
spaghetti with parmesan and black pepper.
""",
    "atomic-habits.md": """# Book Notes: Atomic Habits (James Clear)

date: 2026-05-18
tags: books, productivity, habits

Core claim: identity change drives lasting habit change more reliably than
outcome-based goals. The four laws of behavior change: make it obvious,
make it attractive, make it easy, make it satisfying. Habit stacking:
attach a new habit to an existing one.
""",
    "side-project-ideas.md": """# Side Project Ideas

date: 2026-07-10
tags: projects, ideas, python

Running list, add as they come, prune quarterly. MCP server for these
exact notes -- expose this folder to Claude Desktop as searchable tools
instead of grepping it by hand. (Building this one now.)
""",
}

for name, text in sample_notes.items():
    (NOTES_DIR / name).write_text(text, encoding="utf-8")

print(f"Wrote {len(sample_notes)} sample notes to {NOTES_DIR}")

## The tool functions, as plain Python

Same logic as `server.py` in [`examples/mcp-notes-server/`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/mcp-notes-server/server.py), minus the `@mcp.tool()` decorator and the `FastMCP` server wrapper -- these are just ordinary functions here, callable directly, with no protocol layer between you and the code.

In [ ]:
import time
from dataclasses import dataclass


@dataclass
class Note:
    path: Path
    title: str
    body: str
    modified: float


def _load_note(path: Path) -> Note:
    text = path.read_text(encoding="utf-8", errors="ignore")
    title = path.stem
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith("# "):
            title = stripped[2:].strip()
            break
    return Note(path=path, title=title, body=text, modified=path.stat().st_mtime)


def _all_notes() -> list[Note]:
    if not NOTES_DIR.exists():
        return []
    return [_load_note(p) for p in sorted(NOTES_DIR.glob("*.md"))]


def search_notes(query: str) -> str:
    """Search every note for a keyword and report which notes mention it.

    Same logic the real MCP tool runs -- just called directly here, instead
    of through the MCP protocol.
    """
    query_lower = query.lower()
    matches = []
    for note in _all_notes():
        for line in note.body.splitlines():
            if query_lower in line.lower():
                matches.append(f'"{note.title}": {line.strip()[:160]}')
                break
    if not matches:
        return f"No notes mention '{query}'."
    return "Found in:\n" + "\n".join(matches)


def get_note_by_title(title: str) -> str:
    """Return the full text of one note, matched by exact or partial title."""
    title_lower = title.lower()
    notes = _all_notes()

    exact = [n for n in notes if n.title.lower() == title_lower]
    if len(exact) == 1:
        return exact[0].body

    partial = [n for n in notes if title_lower in n.title.lower()]
    if len(partial) == 1:
        return partial[0].body
    if len(partial) > 1:
        titles = ", ".join(f'"{n.title}"' for n in partial)
        return f"Multiple notes match '{title}': {titles}. Be more specific."

    return f"No note titled '{title}' found."


def list_recent_notes(limit: int = 5) -> str:
    """List the most recently modified notes, newest first."""
    notes = sorted(_all_notes(), key=lambda n: n.modified, reverse=True)[:limit]
    if not notes:
        return "No notes found."

    now = time.time()
    lines = []
    for note in notes:
        age_days = (now - note.modified) / 86400
        age = "today" if age_days < 1 else f"{int(age_days)} days ago"
        lines.append(f'"{note.title}" ({age})')
    return "\n".join(lines)

## Try `search_notes`

Call it exactly like a normal Python function -- no server, no client, no protocol.

In [ ]:
print(search_notes("garlic"))

In [ ]:
print(search_notes("habit"))

## Try `get_note_by_title`

Partial titles work as long as they're unambiguous.

In [ ]:
print(get_note_by_title("pasta"))

## Try `list_recent_notes`

In [ ]:
print(list_recent_notes(limit=3))

## What's missing here -- and where to get it

This notebook only ever calls these three functions directly, in the same Python process, the same way you'd call any other function. It does **not**:

- register them as MCP tools with `@mcp.tool()` and a `FastMCP` server object,
- run a persistent local server process over stdio,
- or connect anything to Claude Desktop (or any other MCP client).

That's the actual point of the project -- a real, standalone server any MCP-compatible client can plug into, indexing a real folder of your own notes. For that, follow the [Build an MCP Server for Your Notes lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/mcp-notes-server) locally with `uv`, using [`examples/mcp-notes-server/server.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/mcp-notes-server/server.py) (which wraps these exact same functions with `@mcp.tool()`) as your starting point.